<a href="https://colab.research.google.com/github/Harshithpalan/Python-projects/blob/main/3D%20Image%20Reconstruction%20from%202D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3D Image Reconstruction from 2D
This notebook demonstrates how to convert a 2D image into a 3D point cloud using:
1. **Deep Learning**: MiDaS for monocular depth estimation.
2. **Geometry**: Back-projecting 2D pixels to 3D coordinates using camera intrinsic assumptions.

In [1]:
!pip install timm open3d

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 71.7 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.7.1
    Uninstalling ipywidgets-7.7.1:
      Successfully uninstalled ipywidgets-7.7.1


In [2]:
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Load MiDaS model for depth estimation
model_type = "MiDaS_small"
midas = torch.hub.load("intel-isl/MiDaS", model_type)
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
midas.to(device)
midas.eval()

# Load transforms
midas_transforms = torch.hub.load("intel-isl/MiDaS", "transforms")
transform = midas_transforms.small_transform if model_type == "MiDaS_small" else midas_transforms.dpt_transform

/usr/local/lib/python3.12/dist-packages/torch/hub.py:247: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to load(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  _check_repo_is_trusted(


Downloading: "https://github.com/intel-isl/MiDaS/zipball/master" to /root/.cache/torch/hub/master.zip


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Loading weights:  None


/usr/local/lib/python3.12/dist-packages/torch/hub.py:247: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to load(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  _check_repo_is_trusted(


Downloading: "https://github.com/rwightman/gen-efficientnet-pytorch/zipball/master" to /root/.cache/torch/hub/master.zip
Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-weights/tf_efficientnet_lite3-b733e338.pth" to /root/.cache/torch/hub/checkpoints/tf_efficientnet_lite3-b733e338.pth
Downloading: "https://github.com/isl-org/MiDaS/releases/download/v2_1/midas_v21_small_256.pt" to /root/.cache/torch/hub/checkpoints/midas_v21_small_256.pt


100%|██████████| 81.8M/81.8M [00:01<00:00, 80.7MB/s]
Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master


In [3]:
import cv2
import torch

def get_depth_map(img_path):
    # Load image using OpenCV
    img = cv2.imread(img_path)
    if img is None:
        raise FileNotFoundError(f"Could not load image at {img_path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Prepare input for MiDaS
    input_batch = transform(img).to(device)

    with torch.no_grad():
        prediction = midas(input_batch)

        prediction = torch.nn.functional.interpolate(
            prediction.unsqueeze(1),
            size=img.shape[:2],
            mode="bicubic",
            align_corners=False,
        ).squeeze()

    output = prediction.cpu().numpy()
    return img, output

### 2D to 3D Projection
Once we have the depth map, we treat each pixel $(u, v)$ with depth $d$ as a 3D point using assumed camera intrinsics:
$x = (u - c_x) \cdot d / f_x$
$y = (v - c_y) \cdot d / f_y$
$z = d$

In [2]:
import open3d as o3d

# 1. Download an example image
!wget -O input_image.jpg https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg

# 2. Get depth map using the previously defined function
rgb_img, depth_map = get_depth_map('input_image.jpg')

# 3. Create Open3D RGBD image
# MiDaS outputs relative inverse depth; we normalize it for visualization
depth_map_norm = (depth_map - depth_map.min()) / (depth_map.max() - depth_map.min())
depth_o3d = o3d.geometry.Image((depth_map_norm * 1000).astype(np.uint16))
color_o3d = o3d.geometry.Image(rgb_img)

rgbd_image = o3d.geometry.RGBDImage.create_from_color_and_depth(
    color_o3d, depth_o3d, convert_rgb_to_intensity=False
)

# 4. Define Camera Intrinsics (standard approximations)
h, w = rgb_img.shape[:2]
intrinsic = o3d.camera.PinholeCameraIntrinsic(
    w, h, w, h, w/2, h/2
)

# 5. Create Point Cloud
pcd = o3d.geometry.PointCloud.create_from_rgbd_image(rgbd_image, intrinsic)

# Flip the point cloud to orient it correctly for visualization
pcd.transform([[1, 0, 0, 0], [0, -1, 0, 0], [0, 0, -1, 0], [0, 0, 0, 1]])

# 6. Visualize 2D results
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.title("Original 2D Image")
plt.imshow(rgb_img)
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title("Estimated Depth Map")
plt.imshow(depth_map, cmap='magma')
plt.axis('off')
plt.show()

print(f"Generated 3D Point Cloud with {len(pcd.points)} points.")

--2026-04-25 04:42:46--  https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 661378 (646K) [image/jpeg]
Saving to: ‘input_image.jpg’

input_image.jpg     100%[===================>] 645.88K  --.-KB/s    in 0.1s    

2026-04-25 04:42:47 (5.93 MB/s) - ‘input_image.jpg’ saved [661378/661378]



NameError: name 'cv2' is not defined